# Quantum neural networks with and without qang

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Viny2030/qang/blob/main/notebooks/qang_qml.ipynb)

This notebook uses the published library (`pip install qang`, module `qang.qml`). A weight-conserving QNN keeps its data in one Hamming-weight sector, so the **qg filter** can keep only the shots that stayed in that sector. Every result below is shown **with qang** (filtered readout) and **without qang** (raw readout), with their difference.

What to expect (RESEARCH_NOTES §75–§80): no quantum advantage (these models are classically simulable); trained on a simulator and run under T1 relaxation, qang recovers the noiseless accuracy exactly; trained under the calibrated noise, a model without qang catches up; dephasing and unequal T1 are not corrected by the filter.

## 1. Install

In [1]:
import importlib, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "qang>=0.6.16", "scikit-learn"], check=False)
importlib.invalidate_caches()
import qang.qml
if not hasattr(qang.qml, "MultiClassQNN"):  # multiclass is on GitHub before the next PyPI release
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps",
                    "git+https://github.com/Viny2030/qang.git"], check=True)
    print("installed qang from GitHub; if an import fails below, restart the runtime once")
import qang
print('qang', qang.__version__)

qang 0.6.15


## 2. Data: breast cancer, 4 PCA features scaled to [-1, 1]

In [2]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from qang.qml import WeightQNN, compare_qang, kept_fraction

d = load_breast_cancer()
rng = np.random.default_rng(0)
idx = np.concatenate([rng.choice(np.where(d.target == c)[0], 100, replace=False) for c in (0, 1)])
Xtr, Xte, ytr, yte = train_test_split(d.data[idx], d.target[idx], test_size=0.3, stratify=d.target[idx], random_state=0)
sc = StandardScaler().fit(Xtr); pca = PCA(4, random_state=0).fit(sc.transform(Xtr))
Ztr, Zte = pca.transform(sc.transform(Xtr)), pca.transform(sc.transform(Xte))
lo, hi = Ztr.min(0), Ztr.max(0)
scale = lambda Z: np.clip(2 * (Z - lo) / (hi - lo) - 1, -1, 1)
Xtr, Xte = scale(Ztr), scale(Zte)
print(Xtr.shape, Xte.shape)

def show(rows, cols=('reading', 'with qang', 'without qang', 'difference')):
    print(f"{cols[0]:<34}{cols[1]:>11}{cols[2]:>13}{cols[3]:>12}")
    for name, q, r in rows:
        print(f"{name:<34}{q:>11.3f}{r:>13.3f}{q - r:>+12.3f}")

(140, 4) (60, 4)


## 3. Weight-1 QNN, trained without noise, run under T1 (gamma = 0.08)

With qang the readout is exactly the noiseless one (F1); the price is the discarded shots, known in advance: (1 - gamma)^(weight x depth).

In [3]:
GAMMA = 0.08
m1 = WeightQNN(n_qubits=5, weight=1).fit(Xtr, ytr, epochs=120, seed=0)
exact1 = m1.score(Xte, yte)
show([("weight 1, T1, trained clean", m1.score(Xte, yte, gamma=GAMMA, qang=True), m1.score(Xte, yte, gamma=GAMMA, qang=False))])
print(f"noiseless accuracy {exact1:.3f}; kept fraction with qang {kept_fraction(GAMMA, 1, m1.depth):.3f}")

reading                             with qang without qang  difference
weight 1, T1, trained clean             0.950        0.933      +0.017
noiseless accuracy 0.950; kept fraction with qang 0.472


## 4. Weight-2 QNN: the effect of qang grows with the weight

A weight-2 state decays twice as fast, so without qang the readout is more biased; with qang it is still exact.

In [4]:
m2 = WeightQNN(n_qubits=5, weight=2).fit(Xtr, ytr, epochs=120, seed=0)
exact2 = m2.score(Xte, yte)
show([("weight 2, T1, trained clean", m2.score(Xte, yte, gamma=GAMMA, qang=True), m2.score(Xte, yte, gamma=GAMMA, qang=False))])
print(f"noiseless accuracy {exact2:.3f}; kept fraction with qang {kept_fraction(GAMMA, 2, m2.depth):.3f}")

reading                             with qang without qang  difference
weight 2, T1, trained clean             0.917        0.900      +0.017
noiseless accuracy 0.917; kept fraction with qang 0.223


## 5. Noise-aware training: training under the noise, with and without qang

If the device noise is known and the model is trained under it, the model without qang learns the decay and catches up (§79–§80). This takes a couple of minutes.

In [5]:
r = compare_qang(WeightQNN(5, 2), Xtr, ytr, Xte, yte, gamma=GAMMA, epochs=120, seed=0, noise_aware=True)
show([("weight 2, T1, trained under noise", r["with qang"], r["without qang"])])

reading                             with qang without qang  difference
weight 2, T1, trained under noise       0.917        0.933      -0.017


## 6. What the filter does not correct: unequal T1 across qubits, and dephasing

In [6]:
unequal = [0.04, 0.06, 0.08, 0.10, 0.12]
show([
    ("weight 1, unequal T1", m1.score(Xte, yte, gamma=unequal, qang=True), m1.score(Xte, yte, gamma=unequal, qang=False)),
    ("weight 1, T1 + dephasing 0.03", m1.score(Xte, yte, gamma=GAMMA, dephasing=0.03, qang=True), m1.score(Xte, yte, gamma=GAMMA, dephasing=0.03, qang=False)),
    ("weight 2, unequal T1", m2.score(Xte, yte, gamma=unequal, qang=True), m2.score(Xte, yte, gamma=unequal, qang=False)),
    ("weight 2, T1 + dephasing 0.03", m2.score(Xte, yte, gamma=GAMMA, dephasing=0.03, qang=True), m2.score(Xte, yte, gamma=GAMMA, dephasing=0.03, qang=False)),
])
print(f"noiseless: weight 1 {exact1:.3f}, weight 2 {exact2:.3f}")

reading                             with qang without qang  difference
weight 1, unequal T1                    0.933        0.933      +0.000
weight 1, T1 + dephasing 0.03           0.800        0.750      +0.050
weight 2, unequal T1                    0.917        0.833      +0.083
weight 2, T1 + dephasing 0.03           0.567        0.683      -0.117
noiseless: weight 1 0.950, weight 2 0.917


## 7. Finite shots (200 per input)

In [7]:
show([
    ("weight 1, T1, 200 shots", m1.score(Xte, yte, gamma=GAMMA, qang=True, shots=200, seed=1), m1.score(Xte, yte, gamma=GAMMA, qang=False, shots=200, seed=1)),
    ("weight 2, T1, 200 shots", m2.score(Xte, yte, gamma=GAMMA, qang=True, shots=200, seed=1), m2.score(Xte, yte, gamma=GAMMA, qang=False, shots=200, seed=1)),
])

reading                             with qang without qang  difference
weight 1, T1, 200 shots                 0.967        0.950      +0.017
weight 2, T1, 200 shots                 0.867        0.867      +0.000


## 8. Multiclass: each qubit is a class (§110)

`MultiClassQNN` reads C classes from one excitation spread over the qubits. Readout `"qubit"`: the score of class c is the excitation probability of qubit c. Readout `"head"`: a linear softmax head on the five qg_Z. The filter keeps the shots with exactly one excitation, so the class probabilities are exactly the noiseless ones under equal T1. Digits 0-4 (5 classes, one per qubit), 4 PCA features. Over 15 runs (§110) the filter added 5.3 points (qubit readout) and 9.1 (head) on average, and 15 and 18 on digits; one split, like the one below, varies from run to run. On 3-class iris the difference is small.

In [8]:
from sklearn.datasets import load_digits
from sklearn.preprocessing import MinMaxScaler
from qang.qml import MultiClassQNN
dd = load_digits(); keep = dd.target < 5
Ar, Ae, br, be = train_test_split(dd.data[keep], dd.target[keep], test_size=0.3, stratify=dd.target[keep], random_state=0)
sd = StandardScaler().fit(Ar); pc = PCA(4, random_state=0).fit(sd.transform(Ar))
Ar, Ae = pc.transform(sd.transform(Ar)), pc.transform(sd.transform(Ae))
mm = MinMaxScaler((-1, 1)).fit(Ar); Ar, Ae = mm.transform(Ar), np.clip(mm.transform(Ae), -1, 1)
rows = []
for ro in ("qubit", "head"):
    mc = MultiClassQNN(n_qubits=5, n_classes=5, readout=ro).fit(Ar, br, epochs=120, seed=0)
    print(f"readout {ro}: noiseless accuracy {mc.score(Ae, be):.3f}")
    rows += [(f"{ro}: T1, trained clean", mc.score(Ae, be, gamma=GAMMA, qang=True), mc.score(Ae, be, gamma=GAMMA, qang=False)),
             (f"{ro}: unequal T1", mc.score(Ae, be, gamma=unequal, qang=True), mc.score(Ae, be, gamma=unequal, qang=False)),
             (f"{ro}: T1, 200 shots", mc.score(Ae, be, gamma=GAMMA, qang=True, shots=200, seed=1),
              mc.score(Ae, be, gamma=GAMMA, qang=False, shots=200, seed=1))]
show(rows)

readout qubit: noiseless accuracy 0.867


readout head: noiseless accuracy 0.930
reading                             with qang without qang  difference
qubit: T1, trained clean                0.867        0.852      +0.015
qubit: unequal T1                       0.845        0.819      +0.026
qubit: T1, 200 shots                    0.841        0.827      +0.015
head: T1, trained clean                 0.930        0.889      +0.041
head: unequal T1                        0.930        0.871      +0.059
head: T1, 200 shots                     0.934        0.878      +0.055


## Summary

* **Trained on a simulator, run under T1:** qang is decisive, exact under equal T1, and more so at higher weight.
* **Trained under the calibrated noise:** with and without qang are within about one test sample.
* **Cost:** a known fraction of the shots, 1 - (1 - gamma)^(weight x depth).
* **Limits:** unequal T1 and dephasing are not corrected; no quantum advantage.

Details, pre-registered predictions and failures: `RESEARCH_NOTES.md` §75–§80, scripts in `examples/qnn_*_qg.py`.